In [ ]:
from isatools.model import (
    Comment,
    Investigation,
    Study,
    StudyFactor,
    FactorValue,
    OntologyAnnotation,
    Characteristic,
    OntologySource,
    Material,
    Sample,
    Source,
    Protocol,
    ProtocolParameter,
    ProtocolComponent,
    ParameterValue,
    Process,
    Publication,
    Person,
    Assay,
    DataFile,
    plink
)

import os
import json
from json import load
import datetime
import isatools
from isatools.convert import isatab2json
from isatools import isajson, isatab



from isatools.convert.json2jsonld import ISALDSerializer

In [ ]:
ptx_tab_path =  "../../../tests/data/tab/PTX-isatab"
currentpath=os.getcwd()
new_ptx_path=os.path.abspath(os.path.join(currentpath, ptx_tab_path))
print(new_ptx_path)
new_config_dir=os.path.abspath(os.path.join(currentpath, "../../../tests/data/configs/xml/MetaboLightsConfig20150707"))

In [ ]:
validation_report = isatab.validate(open(os.path.join(new_ptx_path, 'i_investigation_PTX_Pilot_Daphnia_magna.txt')))

In [ ]:
print(validation_report)

In [ ]:
!pip show isatools

In [ ]:
!which python

In [ ]:


isa_json = isatab2json.convert(ptx_tab_path,use_new_parser=True)
          
with open(os.path.join(ptx_tab_path, 'isa-ptx.json'), 'w') as out_fp:
            json.dump(isa_json, out_fp)

In [ ]:
with  open(os.path.join(ptx_tab_path, 'isa-ptx.json')) as fp:
            report = isajson.validate(fp)

In [ ]:
print(report)

In [ ]:
with  open(os.path.join(ptx_tab_path, 'i_investigation_PTX_Pilot_Daphnia_magna.json')) as fp:
            report = isajson.validate(fp)

In [ ]:
print(report)

In [ ]:
!pip show jupyter

In [ ]:
mtbls_accnums= ["MTBLS718", "MTBLS1820", "MTBLS1846", "MTBLS2289", "MTBLS3563"]

mtbls_accnum=mtbls_accnums[1]

currentpath=os.getcwd()
print(currentpath)
main_path = "../../tests/data/isatab/mtbls/" + mtbls_accnum
new_path=os.path.abspath(os.path.join(currentpath, "../../../tests/data/mtbls/" + mtbls_accnum))
new_config_dir=os.path.abspath(os.path.join(currentpath, "../../../tests/data/configs/xml/MetaboLightsConfig20150707")) #isaconfig-default_v2015-07-02
print(new_path)
data_path = "./notebook-output/" + mtbls_accnum

In [ ]:
isa_json = isatab2json.convert(new_path, use_new_parser=True, config_dir=new_config_dir)
          
with open(os.path.join(new_path, 'isa.json'), 'w') as out_fp:
            json.dump(isa_json, out_fp)

In [ ]:
instance_path = os.path.join(os.getcwd(),new_path, "isa.json")
print(instance_path)

with open(instance_path, 'r') as instance_file:
        instance = load(instance_file)
        instance_file.close()

In [ ]:
# we now invoke the ISALDSerializer function

ontology = "wd"

serializer = ISALDSerializer(instance)
serializer.set_ontology(ontology)
serializer.set_instance(instance)

jsonldcontent = serializer.output

In [ ]:
isa_json_ld_path = os.path.join(os.getcwd(),new_path, mtbls_accnum + "-isa-rdf-" + ontology + ".json")
print(isa_json_ld_path)

with open(isa_json_ld_path, 'w') as outfile:
    json.dump(jsonldcontent,  outfile, ensure_ascii=False, indent=4)

In [ ]:
from rdflib import Graph
from urllib import error

In [ ]:
graph = Graph()
try:
    graph.parse(isa_json_ld_path, format="json-ld")
except error.HTTPError as he:
   print("ERROR:", he, "-", he.with_traceback, ":=:", he.url)

print(graph)

In [ ]:
 with open(os.path.join(os.getcwd(), new_path, mtbls_accnum + "-isa-rdf-" + ontology + ".ttl"), "w") as TTLout:
     graph.serialize(os.path.join(os.getcwd(), new_path, mtbls_accnum + "-isa-rdf-" + ontology + ".ttl"), format='ttl')

In [ ]:
rdf_input=os.path.join(os.getcwd(), new_path, mtbls_accnum + "-isa-rdf-" + ontology + ".ttl")
print(rdf_input)


In [ ]:
graph1=Graph()
try:
    graph1.parse(rdf_input, format='ttl')
except error.HTTPError as he:
   print("ERROR:", he, "-", he.with_traceback, ":=:", he.url)

print(graph1)

In [ ]:
qres = graph1.query("""
SELECT DISTINCT  ?file ?filename WHERE {  
?file a wd:Q5227290 . 
?file wdp:570 ?filename .
}""")

for row in qres:
    print(f"{row.file} | {row.filename}")

### TODO: Review SPARQL queries

In [ ]:


qres = graph.query("""
SELECT DISTINCT  ?person_iri ?lastname ?firstname ?email ?affiliation ?role WHERE {  
?person_iri a wd:Q215627 .
?person_iri wdp:P735 ?firstname .
?person_iri wdp:P734 ?lastname .
?person_iri wdp:P968 ?email .
?person_iri wdp:P361 ?affiliation .
# ?person_iri wdp:P2868 ?role_iri .
?role_iri wdp:P527 ?role .
FILTER (str(?role) = "principal investigator role" ) .
}
""")

for row in qres:
    print(f"{row.firstname} {row.lastname} | {row.email} |  {row.affiliation} | {row.role}")

In [ ]:
qres = graph.query("""
SELECT DISTINCT  ?factor_iri ?name ?type WHERE {  
?factor_iri a wd:Q2487752 .
?factor_iri wd:P1448-FACTORNAME ?name .
#?factor_iri wdp:P527 ?type .

}""")

for row in qres:
    print(f" {row.name} | {row.type}")

In [ ]:
qres = graph.query("""
SELECT DISTINCT  ?study_iri ?study_title ?publication_iri ?pmid ?doi ?name ?authorlist ?title WHERE {
?study_iri a wd:Q101965 .
?study_iri wdp:Q853614 ?study_id .
?study_iri wdp:P1476 ?study_title .
?study_iri wdp:P527 ?publication_iri .
?publication_iri wdp:P1476 ?title .
?publication_iri wdp:P527 ?doi .
?publication_iri wdp:P527 ?pmid .
?publication_iri wdp:P527 ?authorlist .
}""")


for row in qres:
    print(f"  {row.study_title}  |  {row.publication_iri} |  {row.title}  | {row.pmid} | {row.doi} |  {row.authorlist}")